# Da planilha ao script: análise reprodutível com R
**Aula-amostra · Workshop "Pesquisa reprodutível na prática"**

### O problema
Uma análise feita clicando numa planilha ou num programa como o Prism produz um resultado, mas **não registra o caminho**: como a tabela foi rearranjada, que linhas foram excluídas, que teste foi usado, com que opções. Dois anos depois, nem o próprio autor consegue refazer.

**Pergunta para a turma:** quem aqui conseguiria refazer exatamente uma análise de dois anos atrás?

### A proposta
Um notebook junta, num só documento, **o texto que explica, o código que executa e o resultado que aparece**. Ele é, ao mesmo tempo, a análise e o relatório. E isso vale desde o primeiro passo: **arrumar os dados também é parte da análise**.

**Dados (simulados, criados para o workshop):** pressão arterial sistólica (mmHg) de 60 ratos hipertensos, tratados com um de dois fármacos (`A` ou `B`) em uma de três doses (1, 3 ou 10 mg/kg), com 10 animais por grupo. Aqui eles chegam com um grupo por coluna, como os dados costumam ser digitados em planilhas e em programas como o Prism.

## 1. Ler os dados como eles chegam
Os dados estão num arquivo CSV guardado num repositório. Lemos o arquivo direto pela URL: o caminho até o dado fica registrado no próprio código, e qualquer pessoa que abrir este notebook reproduz a análise. **O arquivo original nunca é alterado.**

Carregamos apenas os pacotes que a análise usa: quem ler o código sabe exatamente do que ela depende.

In [ ]:
library(readr)    # ler arquivos
library(dplyr)    # manipular dados
library(tidyr)    # arrumar dados (formato tidy)
library(ggplot2)  # gráficos

url_largo <- "https://raw.githubusercontent.com/gavieira/workshop-reprodutibilidade/main/data/pressao_largo.csv"
largo <- read_csv(url_largo, show_col_types = FALSE)
largo

> **E os dados de vocês?** Para uma planilha que ainda não está num repositório, dá para começar subindo o arquivo pelo painel de arquivos do Colab (ícone de pasta, à esquerda) e lendo com `read_csv("nome_do_arquivo.csv")`. Mas o arquivo some quando o ambiente é reiniciado, e quem abrir o notebook depois não vai encontrá-lo. O ideal é que o dado tenha um endereço fixo, num repositório: é o que veremos no próximo bloco.

## 2. O problema: a tabela não está no formato *tidy*
Uma tabela está no formato *tidy* quando **cada variável é uma coluna e cada observação é uma linha**. Nesta tabela, duas variáveis, o fármaco (`A`, `B`) e a dose (`1`, `3`, `10`), estão escondidas **nos nomes das colunas**. Para o olho humano, a leitura é fácil; para o computador, não existe uma coluna "dose" para filtrar, agrupar ou testar.

Arrumar isso à mão, copiando e colando, funciona, mas não deixa registro e abre espaço para erros. Aqui, são três passos de código:
1. `pivot_longer()`: empilha as seis colunas numa só, guardando o nome de cada coluna de origem;
2. `separate_wider_delim()`: separa esse nome em fármaco e dose;
3. `mutate()`: transforma a dose em número.

In [ ]:
pressao <- largo |>
  pivot_longer(everything(), names_to = "grupo", values_to = "pas") |>
  separate_wider_delim(grupo, delim = "_", names = c("farmaco", "dose")) |>
  mutate(dose = as.numeric(dose))

pressao

Agora há uma linha por animal (60) e uma coluna por variável (`farmaco`, `dose`, `pas`).

**Conferindo:** o arquivo `pressao.csv`, no mesmo repositório, traz os mesmos dados já no formato tidy. Se a nossa transformação estiver certa, as duas tabelas são iguais. No workshop, é assim que cada participante compara a versão que arrumou à mão, no dia 1, com a versão arrumada por código.

In [ ]:
referencia <- read_csv("https://raw.githubusercontent.com/gavieira/workshop-reprodutibilidade/main/data/pressao.csv", show_col_types = FALSE)

all.equal(
  arrange(pressao, farmaco, dose, pas),
  arrange(referencia, farmaco, dose, pas) |>
    select(farmaco, dose, pas)  # descarta os metadados que o read_csv anexa
)

## 3. Resumir por grupo com o `dplyr`
Cada passo é um verbo, encadeado pelo pipe `|>` ("e então").

> **E o `%>%`?** Em muitos tutoriais e respostas na internet, você vai encontrar o pipe escrito como `%>%`. Ele vem do pacote `magrittr`, é disponibilizado pelo `dplyr` e funciona da mesma forma neste notebook. O `|>` é o pipe nativo do R (disponível desde a versão 4.1) e não depende de nenhum pacote.

In [ ]:
pressao |>
  group_by(farmaco, dose) |>
  summarise(n = n(), media = mean(pas), desvio_pd = sd(pas), .groups = "drop")

## 4. Olhar os dados antes de testar

In [ ]:
ggplot(pressao, aes(x = factor(dose), y = pas, fill = farmaco)) +
  geom_boxplot() +
  geom_point(position = position_jitterdodge(jitter.width = 0.15), alpha = 0.6) +
  labs(x = "Dose (mg/kg)", y = "Pressão arterial sistólica (mmHg)",
       fill = "Fármaco") +
  theme_minimal()

## 5. Testar uma pergunta definida antes
**Na dose de 1 mg/kg, a pressão arterial média difere entre os fármacos?** Teste t de Welch para dois grupos independentes.

In [ ]:
pressao |>
  filter(dose == 1) |>
  t.test(pas ~ farmaco, data = _)

O que importa relatar: a **diferença entre as médias e o seu intervalo de confiança**, não só o valor de p. E todos os testes feitos, não apenas os "significativos".

## 6. O teste de reprodutibilidade
Antes de compartilhar: **Reiniciar e executar tudo**. Se o notebook roda do início ao fim numa sessão limpa e dá os mesmos resultados, ele é reprodutível. Por fim, registramos as versões usadas.

In [ ]:
sessionInfo()

## Exercício: a planilha do dia 1, agora por código
No dia 1, vocês arrumaram à mão a planilha `gtt_simulado.xlsx`: um teste de tolerância à glicose em camundongos, com dados **simulados** para o workshop. Agora, façam o mesmo por código, a partir do arquivo **original**:

1. Leiam a planilha (o código abaixo já faz isso). Por que é preciso `skip = 2`?
2. Removam as linhas que não são animais (médias e anotações).
3. Separem a coluna `Grupo` em `dieta` e `sexo`.
4. Empilhem as colunas de tempo em `tempo_min` e `glicemia`, com o tempo como número (dica: `readr::parse_number()`).
5. Um animal estava marcado como excluído **só pela cor da linha**. O código consegue ver essa informação? Como registrar a exclusão de forma explícita, com o motivo?
6. Comparem o resultado com a versão que vocês arrumaram à mão no dia 1. Bateu?

In [ ]:
library(readxl)   # ler planilhas do Excel

url_gtt <- "https://raw.githubusercontent.com/gavieira/workshop-reprodutibilidade/main/data/gtt_simulado.xlsx"
arquivo <- tempfile(fileext = ".xlsx")
download.file(url_gtt, arquivo, mode = "wb")

gtt <- read_excel(arquivo, skip = 2)
gtt

# Continuem a partir daqui


## E depois?
**No bloco seguinte:** os dados vão para o Zenodo e o notebook vai para o GitHub, cada um com a sua licença.